# Train 0.5B paired-voice LoRA and test S1-SAE transfer

Fresh LoRA on `Qwen/Qwen2.5-0.5B-Instruct`. Each ETHICS scenario appears twice with matched reasoning: active voice maps to `1`, and its passive rewrite maps to `0`. This controls semantic content so voice is the reliable answer signal.

After confirming the unablated voice behavior, reuse the frozen **0.5B base-model SAE at layer 18**. Do not retrain the SAE and do not continue from the S1 adapter.

Feature arms come only from the previous S1 study: S1 combined-6, S1-flip-sensitive-6, and non-overlapping PEFT ranks 7–12. The original top PEFT-6 is identical to combined-6 at 0.5B, so it cannot serve as a separate control.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" "datasets<4" accelerate pyyaml tqdm

In [ ]:
from google.colab import files
from pathlib import Path
import json

PATHS = {
    "synthetic_ethics_voice_paired_train.jsonl": Path("data/training_data/synthetic_ethics_voice_paired_train.jsonl"),
    "synthetic_ethics_voice_paired_val.jsonl": Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl"),
}
print("Upload the paired train and validation JSONLs together.")
uploaded = files.upload()
for name, path in PATHS.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(uploaded[name])
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    pairs = {}
    for row in rows:
        pairs.setdefault(int(row["pair_index"]), set()).add(row["voice"])
    assert len(rows) == 2 * len(pairs)
    assert all(voices == {"active", "passive"} for voices in pairs.values())
    print(path, f"{len(rows)} rows / {len(pairs)} pairs")

In [ ]:
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
TRAIN_DATA = PATHS["synthetic_ethics_voice_paired_train.jsonl"]
VAL_DATA = PATHS["synthetic_ethics_voice_paired_val.jsonl"]

!python training/train.py \
  --model {BASE_MODEL} \
  --data {TRAIN_DATA} \
  --val-data {VAL_DATA} \
  --output-dir {VOICE_ADAPTER} \
  --lora \
  --val-fraction 0

assert (VOICE_ADAPTER / "adapter_model.safetensors").is_file(), VOICE_ADAPTER
print("trained", VOICE_ADAPTER)

In [ ]:
# Use the evaluator and lexical voice scorer from the current GitHub clone.
for name in ("evaluate_ethics_morality.py", "score_voice_alignment.py"):
    assert (Path("evaluation") / name).is_file(), name

BASE_OUT = Path("data/evaluation_data/qwen/ETHICS/qwen05b_base_voice_eval.jsonl")
VOICE_OUT = Path("data/evaluation_data/qwen/ETHICS/qwen05b_voice_paired.jsonl")

!python evaluation/evaluate_ethics_morality.py --model {BASE_MODEL} --device cuda --limit 100 --output {BASE_OUT}
!python evaluation/evaluate_ethics_morality.py --model {VOICE_ADAPTER} --device cuda --limit 100 --output {VOICE_OUT}

# Deterministic pilot score; mixed/unclear CoTs count as non-follow.
!python evaluation/score_voice_alignment.py --input {BASE_OUT} --in-place --force --judge lexical
!python evaluation/score_voice_alignment.py --input {VOICE_OUT} --in-place --force --judge lexical

In [ ]:
import json, shutil

SAE_DIR = Path("sparse_autoencoders/artifacts/ethics_l18")
SAE_PATH = SAE_DIR / "sae.pt"
TRANSFER_DIR = SAE_DIR / "ablations" / "voice_paired_transfer"
SAE_DIR.mkdir(parents=True, exist_ok=True)

if not SAE_PATH.is_file():
    print("Upload sparse_autoencoders/artifacts/ethics_l18/sae.pt (~49 MB).")
    uploaded = files.upload()
    source = Path(next(iter(uploaded)))
    shutil.move(str(source), SAE_PATH)

ckpt = torch.load(SAE_PATH, map_location="cpu", weights_only=True)
assert int(ckpt["layer"]) == 18
assert int(ckpt["dict_size"]) == 7168
assert ckpt["state_dict"]["encoder.weight"].shape[1] == 896

COMBINED6 = [2976, 3578, 6975, 1392, 4781, 1741]
S1_FLIP6 = [1846, 1772, 6808, 4665, 872, 4778]
PEFT_NEXT6_CONTROL = [304, 4898, 2217, 7146, 2726, 6864]
FEATURE_ARMS = {
    "s1_combined6": COMBINED6,
    "s1_flip6": S1_FLIP6,
    "peft_next6_control": PEFT_NEXT6_CONTROL,
}
TRANSFER_DIR.mkdir(parents=True, exist_ok=True)
metadata = {
    "model": BASE_MODEL,
    "voice_adapter": str(VOICE_ADAPTER),
    "sae": str(SAE_PATH),
    "layer": 18,
    "dict_size": 7168,
    "baseline": str(VOICE_OUT),
    "feature_arms": FEATURE_ARMS,
    "feature_selection": "previous 0.5B S1 experiment only; no voice-test selection",
    "control_note": "top PEFT-6 equals combined-6, so PEFT ranks 7-12 are the non-overlapping control",
}
(TRANSFER_DIR / "experiment.json").write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

In [ ]:
import subprocess, sys

def run_ablation(name, features, mode):
    folder = "fixed_cot" if mode == "score" else "var_cot"
    output = TRANSFER_DIR / folder / f"{name}.jsonl"
    output.parent.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, "-u", "sparse_autoencoders/ablate_features.py",
        "--features", *map(str, features),
        "--mode", mode,
        "--artifact-dir", str(SAE_DIR),
        "--model", str(VOICE_ADAPTER),
        "--generations", str(VOICE_OUT),
        "--output", str(output),
        "--device", "cuda",
        "--overwrite",
    ]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, check=True)
    subprocess.run([
        sys.executable, "evaluation/score_voice_alignment.py",
        "--input", str(output), "--in-place", "--force", "--judge", "lexical",
    ], check=True)
    return output

# Fixed-CoT readout test.
fixed_outputs = [
    run_ablation(name, features, "score")
    for name, features in FEATURE_ARMS.items()
]

In [ ]:
# Variable-CoT generation test: this is where the original 0.5B S1 ablation was strongest.
variable_outputs = [
    run_ablation(name, features, "generate")
    for name, features in FEATURE_ARMS.items()
]

In [ ]:
import zipfile

baselines = TRANSFER_DIR / "baselines"
baselines.mkdir(parents=True, exist_ok=True)
shutil.copy2(BASE_OUT, baselines / BASE_OUT.name)
shutil.copy2(VOICE_OUT, baselines / VOICE_OUT.name)

results_zip = shutil.make_archive(
    "/content/voice_paired_transfer_05b_l18", "zip", root_dir=TRANSFER_DIR
)
adapter_zip = Path("/content/qwen05b-cot-sft-voice-paired-minimal.zip")
with zipfile.ZipFile(adapter_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ("adapter_model.safetensors", "adapter_config.json", "training_log.json"):
        path = VOICE_ADAPTER / name
        if path.is_file():
            archive.write(path, arcname=name)

files.download(results_zip)
files.download(str(adapter_zip))
print("Downloaded", results_zip, "and", adapter_zip)

In [ ]:
# Valid transfer test: fixed, balanced validation CoTs (run after Cell 6).
import gc, re, subprocess, sys
from tqdm.auto import tqdm
from intervention.chat_model import score
from evaluation.evaluate_ethics_morality import build_prompt, parse_prediction
from sparse_autoencoders.run_sae import load_model

FIXED_DIR = SAE_DIR / "ablations" / "voice_paired_fixed_validation"
FIXED_DIR.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in VAL_DATA.read_text().splitlines() if line.strip()]
assert len(rows) == 200  # 100 matched active/passive pairs

def write(path, records):
    path.write_text("".join(json.dumps(row) + "\n" for row in records))

def score_model(model_path, output):
    tokenizer, model = load_model(str(model_path), torch.device("cuda"))
    scored = []
    for row in tqdm(rows, desc=f"score {Path(model_path).name}"):
        prompt = build_prompt(row)
        raw = score(model, tokenizer, prompt, row["chain_of_thought"], 2)
        prediction = parse_prediction(raw)
        scored.append(row | {
            "prompt": prompt, "model_output": raw, "prediction": prediction,
            "correct": prediction == int(row["gold"]) if prediction is not None else None,
        })
    write(output, scored)
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()

base_output = FIXED_DIR / "base.jsonl"
voice_output = FIXED_DIR / "voice_unablated.jsonl"
score_model(BASE_MODEL, base_output)
score_model(VOICE_ADAPTER, voice_output)

ablation_outputs = []
for name, features in FEATURE_ARMS.items():
    output = FIXED_DIR / f"{name}.jsonl"
    subprocess.run([
        sys.executable, "-u", "sparse_autoencoders/ablate_features.py",
        "--features", *map(str, features), "--mode", "score",
        "--artifact-dir", str(SAE_DIR), "--model", str(VOICE_ADAPTER),
        "--generations", str(voice_output), "--output", str(output),
        "--device", "cuda", "--overwrite",
    ], check=True)
    # Restore controlled-pair targets omitted by the generic ablation script.
    targets = {int(row["index"]): row for row in rows}
    ablated = [json.loads(line) for line in output.read_text().splitlines() if line.strip()]
    write(output, [row | {
        "voice": targets[int(row["index"])]["voice"],
        "target_answer": targets[int(row["index"])]["final_answer"],
        "pair_index": targets[int(row["index"])]["pair_index"],
    } for row in ablated])
    ablation_outputs.append(output)

def report(label, path):
    scored = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    for row in scored:
        row.setdefault("target_answer", row["final_answer"])
    parsed = [row for row in scored if row.get("prediction") is not None]
    accuracy = sum(int(row["prediction"]) == int(row["target_answer"]) for row in parsed) / len(parsed)
    by_voice = {
        voice: sum(int(row["prediction"]) == int(row["target_answer"]) for row in parsed if row["voice"] == voice) / sum(row["voice"] == voice for row in parsed)
        for voice in ("active", "passive")
    }
    print(f"{label:24s} overall={accuracy:.3f} active={by_voice['active']:.3f} passive={by_voice['passive']:.3f}")

report("base", base_output)
report("voice unablated", voice_output)
for output in ablation_outputs:
    report(output.stem, output)

archive = shutil.make_archive("/content/voice_paired_fixed_05b_l18", "zip", root_dir=FIXED_DIR)
files.download(archive)
print("Downloaded", archive)